In [1]:
import copy
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np

# -----------------------------
# 1. Simple Neural Network Model
# -----------------------------
class SimpleModel(nn.Module):
    def __init__(self, input_dim=10):
        super(SimpleModel, self).__init__()
        self.fc1 = nn.Linear(input_dim, 32)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(32, 2)

    def forward(self, x):
        x = self.relu(self.fc1(x))
        return self.fc2(x)

# -----------------------------
# 2. Create Synthetic Client Data
# -----------------------------
def create_client_data(num_samples):
    X = torch.randn(num_samples, 10)
    y = torch.randint(0, 2, (num_samples,))
    return TensorDataset(X, y)

# -----------------------------
# 3. Client Local Training
# -----------------------------
def local_train(model, dataset, epochs=2, lr=0.01):
    model.train()
    loader = DataLoader(dataset, batch_size=16, shuffle=True)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=lr)

    for _ in range(epochs):
        for X, y in loader:
            optimizer.zero_grad()
            output = model(X)
            loss = criterion(output, y)
            loss.backward()
            optimizer.step()

    return model.state_dict()

# -----------------------------
# 4. Server-Side Weighted Federated Averaging
# -----------------------------
def weighted_fedavg(global_model, client_states, client_sizes):
    total_samples = sum(client_sizes)
    new_state = copy.deepcopy(global_model.state_dict())

    for key in new_state.keys():
        new_state[key] = torch.zeros_like(new_state[key])

        for client_state, size in zip(client_states, client_sizes):
            weight = size / total_samples
            new_state[key] += client_state[key] * weight

    global_model.load_state_dict(new_state)
    return global_model

# -----------------------------
# 5. Federated Learning Simulation
# -----------------------------
def federated_training(num_clients=3, rounds=5):

    # Initialize Global Model
    global_model = SimpleModel()

    # Create Clients with Different Data Sizes
    client_data_sizes = [100, 200, 300]
    client_datasets = [create_client_data(size) for size in client_data_sizes]

    for round_num in range(rounds):
        print(f"\n🔁 Federated Round {round_num+1}")

        client_states = []

        # Each client receives global model
        for i in range(num_clients):
            local_model = copy.deepcopy(global_model)
            updated_state = local_train(local_model, client_datasets[i])
            client_states.append(updated_state)

        # Server Aggregation
        global_model = weighted_fedavg(
            global_model,
            client_states,
            client_data_sizes
        )

        print("✅ Global model updated and redistributed")

    return global_model


# -----------------------------
# 6. Run Federated Training
# -----------------------------
if __name__ == "__main__":
    final_global_model = federated_training()
    print("\n🎯 Federated Learning Completed Successfully")


🔁 Federated Round 1
✅ Global model updated and redistributed

🔁 Federated Round 2
✅ Global model updated and redistributed

🔁 Federated Round 3
✅ Global model updated and redistributed

🔁 Federated Round 4
✅ Global model updated and redistributed

🔁 Federated Round 5
✅ Global model updated and redistributed

🎯 Federated Learning Completed Successfully
